In [102]:
from backend.app.db.connection import create_connection

from backend.app.repositories.city_repository import (
    CityRepository,
)

from backend.app.repositories.weather_repository import (
    WeatherRepository,
)

from backend.app.services.city_service import (
    CityService,
)

from backend.app.services.weather_service import (
    WeatherService,
)

from backend.app.services.weather_analysis_service import (
    aggregate_hourly_weather_by_day,mean
)

from backend.app.models.weather import (
    WeatherDaily
)



from datetime import date, datetime
from collections import defaultdict

In [103]:
connection = create_connection()

city_repository = CityRepository(connection)

city_service = CityService(connection,city_repository)

In [104]:
cities = city_service.get_cities()

In [105]:
weather_repository = WeatherRepository(connection)

weather_service = WeatherService(connection,weather_repository)

In [106]:
#for city in cities:
    #print(aggregate_hourly_weather_by_day(weather_service.get_hourly_weather(city.id)))

In [107]:
daily_data = aggregate_hourly_weather_by_day(weather_service.get_hourly_weather(1))

#weather data grouped by day of year (month, day)
grouped: dict[
    tuple[int, int],
    list[WeatherDaily],
] = defaultdict(list)

for record in daily_data:
    observed_date = datetime.fromisoformat(
            record.observed_date
        ).date()
    grouped[(observed_date.month, observed_date.day)].append(record)

#print(grouped[(1,1)])

#weather averages for every day of the year
daily_averages: dict[
    tuple[int, int],
    WeatherDaily,
] = {}

for record in sorted(grouped):

    records_list = grouped[record]

    daily_averages[record] = WeatherDaily(
        city_id=records_list[0].city_id,
        observed_date=records_list[0].observed_date[5:],
        temperature_2m_mean=round(mean(item.temperature_2m_mean for item in records_list),2),
        temperature_2m_max=round(mean(item.temperature_2m_max for item in records_list),2),
        temperature_2m_min=round(mean(item.temperature_2m_min for item in records_list),2),
        precipitation_sum=round(mean(item.precipitation_sum for item in records_list),2),
        cloud_cover_mean=round(mean(item.cloud_cover_mean for item in records_list),2),
        relative_humidity_2m_mean=round(mean(item.relative_humidity_2m_mean for item in records_list),2),
        wind_speed_10m_mean=round(mean(item.wind_speed_10m_mean for item in records_list),2),
    )

print(daily_averages)

{(1, 1): WeatherDaily(city_id=1, observed_date='01-01', temperature_2m_mean=-4.6, temperature_2m_max=-2.67, temperature_2m_min=-6.67, precipitation_sum=1.22, cloud_cover_mean=83.82, relative_humidity_2m_mean=87.42, wind_speed_10m_mean=16.16), (1, 2): WeatherDaily(city_id=1, observed_date='01-02', temperature_2m_mean=-5.94, temperature_2m_max=-4.45, temperature_2m_min=-7.62, precipitation_sum=1.42, cloud_cover_mean=81.88, relative_humidity_2m_mean=86.28, wind_speed_10m_mean=15.73), (1, 3): WeatherDaily(city_id=1, observed_date='01-03', temperature_2m_mean=-6.92, temperature_2m_max=-5.19, temperature_2m_min=-8.62, precipitation_sum=1.15, cloud_cover_mean=90.03, relative_humidity_2m_mean=84.76, wind_speed_10m_mean=16.3), (1, 4): WeatherDaily(city_id=1, observed_date='01-04', temperature_2m_mean=-6.83, temperature_2m_max=-4.72, temperature_2m_min=-8.96, precipitation_sum=0.96, cloud_cover_mean=85.18, relative_humidity_2m_mean=86.44, wind_speed_10m_mean=13.91), (1, 5): WeatherDaily(city_id=